### ЗАДАНИЕ 1
Напишите функцию ```get_pretrained_model```, которая принимает в качестве аргументов название архитектуры, количество классов для задачи классификации и стоит ли инициализировать модель с помощью полученных в ходе обучения на датасете ```ImageNet```. Она должна иметь следующую сигнатуру: ```def get_pretrained_model(model_name: str, num_classes: int, pretrained: bool=True):```

Будем считать, что на вход могут прийти четыре различных model_name: ```alexnet```, ```vgg11```, ```googlenet``` и ```resnet18```. Для каждого из них нужно вернуть соответствующую модель из зоопарка моделей ```torchvision```.

Чтобы понять, как именно модифицировать созданные объекты, посмотрите на исходный код для моделей:

1. https://pytorch.org/hub/pytorch_vision_resnet/
2. https://pytorch.org/hub/pytorch_vision_alexnet/
3. https://pytorch.org/hub/pytorch_vision_vgg/
4. https://pytorch.org/hub/pytorch_vision_googlenet/

In [2]:
import torch
import torch.nn as nn
from torchvision.models import (
    alexnet, vgg11, googlenet, resnet18,
    AlexNet_Weights, VGG11_Weights,
    GoogLeNet_Weights, ResNet18_Weights
)

def get_pretrained_model(model_name: str,
                         num_classes: int,
                         pretrained: bool = True,
                         aux_logits: bool = True):

    if model_name == 'alexnet':
        weights = AlexNet_Weights.DEFAULT if pretrained else None
        model = alexnet(weights=weights)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)

    elif model_name == 'vgg11':
        weights = VGG11_Weights.DEFAULT if pretrained else None
        model = vgg11(weights=weights)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)

    elif model_name == 'resnet18':
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == 'googlenet':
        weights = GoogLeNet_Weights.DEFAULT if pretrained else None
        model = googlenet(weights=weights, aux_logits=aux_logits)

        # основной классификатор
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

        # вспомогательные — только если включены
        if aux_logits:
            model.aux1.fc2 = nn.Linear(model.aux1.fc2.in_features, num_classes)
            model.aux2.fc2 = nn.Linear(model.aux2.fc2.in_features, num_classes)

    else:
        raise ValueError(f'{model_name} не поддерживается')

    return model

### 2 задание
Модифицируйте свою лучшую модель из 4-го занятия 10-го задания, добавьте в нее skip-connection. Обучите ее на CIFAR10. Сравните качество со skip-connection и без skip-connection. Добейтесь Accuracy в 88.5% на тестовой выборке.
Вам нужно сдать код с функцией, которая возвращает модель, назовите эту функцию create_advanced_skip_connection_conv_cifar

In [3]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        # блоки архитектуры
        self.block_1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1), ### пишу размер, который стал: 32x32x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(p=0.2)
        )
        
        self.block_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1), ### пишу размер, который стал: 32x32x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(p=0.2)
        )

        self.block_3 = nn.Sequential(
             nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1, dilation=2), ### 30x30x128
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(p=0.2)
        )

        self.block_4 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1, dilation=2), ### 28x28x128
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2), ### 14x14x128
            nn.Dropout2d(p=0.2),
        )

        self.block_5 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=3, dilation=2), ### 16x16x256
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2), ### 8x8x256
            nn.Dropout2d(p=0.2),
        )

        self.block_6 = nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1), ### 8x8x256
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(p=0.2),
        )

        self.block_7 = nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=2, dilation=2), ### 8x8x512
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2), ### 4x4x512
            nn.Dropout2d(p=0.2),
        )

        self.block_8 = nn.Sequential(
            nn.Conv2d(in_channels=512, out_channels=128, kernel_size=3, padding=1), ### 4x4x128
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(p=0.2),
        )

        self.block_9 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1), ### 4x4x128
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(p=0.2),
        )

        self.block_10 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=64, kernel_size=3, padding=1), ### 4x4x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(p=0.2),
        )

        self.fc1 = nn.Sequential(
            nn.Linear(4*4*64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
        )
        self.fc2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
        )
        self.fc3 = nn.Linear(128, 10)

    def forward(self,x):
        x = self.block_1(x)
        skip = x
        x = self.block_2(x)
        x = x + skip
        x = self.block_3(x)
        x = self.block_4(x)
        x = self.block_5(x)
        skip = x
        x = self.block_6(x)
        x = x + skip
        x = self.block_7(x)
        x = self.block_8(x)
        skip = x
        x = self.block_9(x)
        x = x + skip
        x = self.block_10(x)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)

        return x
        

def create_advanced_skip_connection_conf_cifar():
    return Model()  